## Example 1: statsmodels functionality test 1

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

# Example 1: Basic linear regression with synthetic data
print("=== Example 1: Basic Linear Regression ===")

# Generate synthetic dataset
np.random.seed(42)
n_samples = 100
X = np.random.randn(n_samples, 2)
true_coef = np.array([2.5, -1.8])
y = X.dot(true_coef) + 0.5 * np.random.randn(n_samples)

# Add constant term for intercept
X_with_const = sm.add_constant(X)

# Fit OLS regression model
model = sm.OLS(y, X_with_const)
results = model.fit()

print(f"statsmodels version: {sm.__version__}")
print(f"Dataset shape: {X.shape}")
print(f"True coefficients: {true_coef}")

# Display results summary
print(f"\nRegression Results:")
print(f"R-squared: {results.rsquared:.4f}")
print(f"Adjusted R-squared: {results.rsquared_adj:.4f}")
print(f"F-statistic: {results.fvalue:.2f}")
print(f"Prob (F-statistic): {results.f_pvalue:.2e}")

# Show coefficient estimates
print(f"\nCoefficient estimates:")
for i, (coef, pval, ci_low, ci_high) in enumerate(zip(
    results.params, results.pvalues, results.conf_int()[0], results.conf_int()[1]
)):
    var_name = 'const' if i == 0 else f'x{i}'
    print(f"  {var_name}: {coef:.4f} (p={pval:.3f}, CI=[{ci_low:.3f}, {ci_high:.3f}])")

# Model diagnostics
print(f"\nModel diagnostics:")
print(f"AIC: {results.aic:.2f}")
print(f"BIC: {results.bic:.2f}")
print(f"Log-likelihood: {results.llf:.2f}")
print(f"Condition number: {results.condition_number:.1f}")

## Example 2: statsmodels functionality test 2

In [ ]:
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
import numpy as np
import pandas as pd

# Example 2: Time series seasonal decomposition
print("=== Example 2: Time Series Analysis ===")

# Generate time series with weekly seasonality
np.random.seed(123)
dates = pd.date_range('2020-01-01', periods=100, freq='D')

# Trend + weekly seasonal + noise
trend = np.linspace(100, 150, len(dates))
seasonal = 10 * np.sin(2 * np.pi * np.arange(len(dates)) / 7)
noise = np.random.normal(0, 3, len(dates))

ts_data = trend + seasonal + noise
ts_df = pd.DataFrame({'value': ts_data}, index=dates)

print(f"Time series shape: {ts_df.shape}")
print(f"Date range: {ts_df.index.min()} to {ts_df.index.max()}")

# Seasonal decomposition with weekly period
decomposition = seasonal_decompose(ts_df['value'], model='additive', period=7)

print(f"\nDecomposition (period=7):")
print(f"Trend range: {decomposition.trend.min():.2f} to {decomposition.trend.max():.2f}")
print(f"Seasonal range: {decomposition.seasonal.min():.2f} to {decomposition.seasonal.max():.2f}")
print(f"Residual std: {decomposition.resid.std():.2f}")

print("\nExample 2 completed!")

## Example 3: statsmodels functionality test 3

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
import numpy as np
import pandas as pd

# Example 3: Advanced regression with formula API and diagnostics
print("=== Example 3: Advanced Regression Analysis ===")

# Create realistic dataset for economic analysis
np.random.seed(456)
n_obs = 200

# Simulate economic data: house prices based on multiple factors
data = {
    'income': np.random.lognormal(mean=10.8, sigma=0.5, size=n_obs),  # Household income
    'sqft': np.random.normal(1800, 400, n_obs),                      # Square footage
    'age': np.random.exponential(scale=15, size=n_obs),              # House age
    'rooms': np.random.poisson(lam=6, size=n_obs),                  # Number of rooms
    'metro': np.random.binomial(1, 0.6, n_obs),                     # Metro area dummy
    'school_rating': np.random.normal(7.5, 1.5, n_obs)              # School district rating
}

# Generate house prices with realistic relationships
price = (
    0.8 * data['income'] +
    120 * data['sqft'] / 1000 +
    -2000 * data['age'] / 10 +
    8000 * data['rooms'] +
    25000 * data['metro'] +
    12000 * data['school_rating'] +
    np.random.normal(0, 20000, n_obs)
)

data['price'] = np.maximum(price, 50000)  # Ensure positive prices

# Create DataFrame
df = pd.DataFrame(data)

print(f"Dataset shape: {df.shape}")
print(f"\nSummary statistics:")
print(df[['price', 'income', 'sqft', 'age', 'rooms']].describe())

# Fit multiple regression using formula API
formula = 'price ~ income + sqft + age + rooms + metro + school_rating'
model = smf.ols(formula, data=df)
results = model.fit()

print(f"\nRegression Formula: {formula}")
print(f"R-squared: {results.rsquared:.4f}")
print(f"Adjusted R-squared: {results.rsquared_adj:.4f}")
print(f"F-statistic: {results.fvalue:.2f} (p={results.f_pvalue:.2e})")

# Coefficient interpretation
print(f"\nCoefficient Estimates (with interpretation):")
coef_names = {
    'Intercept': 'Base price',
    'income': 'Price increase per $1 income',
    'sqft': 'Price increase per sq ft',
    'age': 'Price decrease per year of age',
    'rooms': 'Price increase per room',
    'metro': 'Metro area premium',
    'school_rating': 'Price increase per rating point'
}

for param, coef in results.params.items():
    pval = results.pvalues[param]
    significance = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    interpretation = coef_names.get(param, param)
    print(f"  {param}: {coef:.2f}{significance} ({interpretation})")

# Advanced model diagnostics
print(f"\nModel Diagnostics:")

# Durbin-Watson test for autocorrelation
dw_stat = durbin_watson(results.resid)
print(f"Durbin-Watson statistic: {dw_stat:.4f}")

# Breusch-Pagan test for heteroscedasticity
bp_stat, bp_pval, _, _ = het_breuschpagan(results.resid, results.model.exog)
print(f"Breusch-Pagan test: statistic={bp_stat:.4f}, p-value={bp_pval:.4f}")

# Residual analysis
residuals = results.resid
print(f"Residual statistics:")
print(f"  Mean: {residuals.mean():.2f}")
print(f"  Std: {residuals.std():.2f}")
print(f"  Min: {residuals.min():.2f}")
print(f"  Max: {residuals.max():.2f}")

# Prediction and confidence intervals
new_data = pd.DataFrame({
    'income': [60000], 'sqft': [2000], 'age': [10], 
    'rooms': [4], 'metro': [1], 'school_rating': [8.0]
})

predictions = results.get_prediction(new_data)
print(f"\nPrediction for new house:")
print(f"Predicted price: ${predictions.predicted_mean[0]:,.0f}")
print(f"95% Confidence interval: ${predictions.conf_int()[0][0]:,.0f} - ${predictions.conf_int()[0][1]:,.0f}")

# Model comparison using information criteria
print(f"\nModel Selection Criteria:")
print(f"AIC: {results.aic:.2f}")
print(f"BIC: {results.bic:.2f}")
print(f"Log-likelihood: {results.llf:.2f}")

## Example 4: statsmodels functionality test 4

In [ ]:
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Logit
import numpy as np
import pandas as pd

# Example 4: Logistic regression
print("=== Example 4: Logistic Regression ===")

# Generate binary outcome data
np.random.seed(789)
n = 200

X = np.random.randn(n, 2)
linear = -1 + 0.5 * X[:, 0] + 1.2 * X[:, 1]
prob = 1 / (1 + np.exp(-linear))
y = np.random.binomial(1, prob)

# Fit logistic regression
X_const = sm.add_constant(X)
model = Logit(y, X_const)
results = model.fit()

print(f"Pseudo R-squared: {results.prsquared:.4f}")
print(f"Log-likelihood: {results.llf:.2f}")

# Odds ratios
print(f"\nOdds ratios: {np.exp(results.params)}")

print("\nExample 4 completed!")

## Example 5: statsmodels functionality test 5

In [ ]:
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
import numpy as np
import pandas as pd

# Example 5: ARIMA time series model
print("=== Example 5: ARIMA Model ===")

# Generate time series data
np.random.seed(999)
n = 200

# AR(1) process
ar_coef = 0.7
ts = np.zeros(n)
ts[0] = np.random.normal()
for t in range(1, n):
    ts[t] = ar_coef * ts[t-1] + np.random.normal()

# Create series
dates = pd.date_range('2020-01-01', periods=n, freq='D')
series = pd.Series(ts, index=dates)

print(f"Time series shape: {series.shape}")
print(f"Mean: {series.mean():.3f}, Std: {series.std():.3f}")

# Fit ARIMA model
model = ARIMA(series, order=(1, 0, 0))
results = model.fit()

print(f"\nARIMA(1,0,0) Results:")
print(f"AIC: {results.aic:.2f}")
print(f"BIC: {results.bic:.2f}")

print("\nExample 5 completed!")